# Real Analysis, Written as in Your Book
### ε–N, ε–δ and the big theorems, with every step checked by Lean

This notebook is for mathematicians who want to write analysis proofs **the way they appear in a textbook
or a paper** — *"Let ε > 0. Choose N such that … Take δ = ε/3. Then …"* — and have each step checked by the
**Lean proof assistant** and its library **Mathlib**.

* You write the proof as a sequence of sentences. Each sentence is one PyLeanX step.
* Lean checks every step from the axioms. A ✅ appears only when the whole proof is correct and the
  axiom audit finds nothing beyond the three standard axioms of Lean's mathematics
  (`propext`, `Classical.choice`, `Quot.sound`).
* When a step does not follow, Lean says **which step**, so you can repair that step only.
* The proof is printed back as numbered prose, with the reason of each step, ready to read.

**Contents:** 1. Phrasebook · 2. Vocabulary and theorem catalogue · 3. Worked examples
(1/(n+1) → 0, uniqueness of limits, ε–δ continuity, Bolzano–Weierstrass in use, citing your own
theorem) · 4. When a step is wrong · 5. Exercises · 6. Writing your own · 7. Solutions

In [1]:
import pyleanx as pl
from pyleanx import analysis as an
from analysis_lecture import *      # statements, figures, check(), the ε–N / ε–δ helpers

# Letters already defined by analysis_lecture:
#   sequences a, b (ℕ → ℝ);  functions f, p (ℝ → ℝ);  φ = phi (ℕ → ℕ, indices of a subsequence)
#   real numbers L, L2 (printed L′), eps (ε), x, x0 (x₀), c;  natural numbers n, m, N, N1, N2
#   R(n): the natural number n as a real number (printed n)

---
## 1. Phrasebook: from the book to PyLeanX

| In the book | In PyLeanX |
|---|---|
| *Let ε > 0.* | `pl.fix(eps, assume={"hε": eps > 0})` |
| *Let x with \|x − x₀\| < δ.* | `pl.fix(x, assume={"hx": pl.abs(x - x0) < delta})` |
| *Choose N such that …* (by a fact) | `pl.choose(N, such_that=..., because=...)` |
| *Take N.* · *Take δ = ε/3 > 0.* | `pl.take(N)` · `pl.take(eps / 3, such_that=eps / 3 > 0)` |
| *It suffices to show P* (by the definition) | `pl.suffices(P, because=an.tendsto_iff)` |
| *We have P* (because …) | `pl.claim(P, because=...)` |
| *Suppose, for contradiction, that P.* | `pl.suppose_not(P)` |
| *\|A\| = \|B\| ≤ C < ε.* (each step with its reason) | `pl.chain(A, ("=", B, reason), ("<=", C, reason), ("<", eps))` |
| *Let p(x) = x³ + x − 1.* | `pl.let(p, lambda t: t**3 + t - 1)` |
| *By the intermediate value theorem …* | `because=an.ivt` |
| *By Theorem 1 …* (your own, already proved) | `because=my_theorem` |
| *Hence … ∎* | `pl.conclude(because=...)` |

**What Lean fills in by itself:** algebra (`ring`), linear inequalities (`linarith`), arithmetic on ℕ
(`omega`), positivity, continuity of explicit formulas, and instantiating a cited statement.
**What you must say:** every analytic fact (a definition, the triangle inequality, a theorem) as a reason.

**Conventions.** A sequence is a function ℕ → ℝ indexed from 0 (so "1/n" is written 1/(n+1)).
`R(n)` is the natural number n viewed as a real number (printed simply as n). `pl.fun(lambda n: …, pl.Nat)`
is the sequence given by a formula.

---
## 2. Vocabulary and theorem catalogue (`pyleanx.analysis`, imported as `an`)

**Definitions** — each comes with its unfolding, to cite when a proof works from the definition:

| Write | Means | Unfold with |
|---|---|---|
| `an.tendsto(a, L)` | $a_n \to L$ | `an.tendsto_iff`: ∀ε>0 ∃N ∀n≥N, \|aₙ − L\| < ε |
| `an.cauchy(a)` | $(a_n)$ is Cauchy | `an.cauchy_iff`: ∀ε>0 ∃N ∀m,n≥N, \|aₘ − aₙ\| < ε |
| `an.continuous_at(f, x0)` | f is continuous at x₀ | `an.continuous_at_iff`: ∀ε>0 ∃δ>0 ∀x, \|x−x₀\|<δ → \|f(x)−f(x₀)\|<ε |
| `an.uniformly_continuous_on(f, a, b)` | uniformly continuous on [a, b] | `an.uniformly_continuous_iff` |
| `an.continuous_on(f, a, b)`, `an.continuous(f)` | continuous on [a, b] / on ℝ | |
| `an.differentiable_on(f, a, b)`, `an.deriv(f, x)` | differentiable on (a, b), f′(x) | |
| `an.increasing(a)`, `an.strictly_increasing(φ)`, `an.bounded(a)` | monotone, strictly monotone, ∃M ∀n \|aₙ\| ≤ M | |

**Theorems you can cite** (each prints as the reason a book gives):

| Chapter | Facts |
|---|---|
| Limits of sequences | `limit_unique`, `limit_add`, `limit_sub`, `limit_mul`, `limit_const_mul`, `limit_const`, `squeeze`, `one_over_n`, `archimedean`, `reciprocal_lt` |
| Completeness | `convergent_bounded`, `convergent_cauchy`, `cauchy_convergent`, `monotone_convergence`, `increasing_of_step`, `bolzano_weierstrass` |
| Continuity | `sequential_continuity`, `ivt`, `evt_max`, `evt_min`, `heine_cantor`, `continuous_on_of_continuous`, `continuous_at_of_continuous` |
| Differentiation | `rolle`, `mvt`, `differentiable_on_of_differentiable`, `constant_of_zero_derivative`, `deriv_sin`, `deriv_cos` |
| Absolute value | `triangle_inequality`, `reverse_triangle`, `abs_sub_comm`, `abs_mul`, `abs_lt`, `abs_le`, `abs_pos`, `abs_of_pos`, `abs_nonneg`, `le_max_left`, `le_max_right` |

The facts not stated this way in Mathlib are shipped as Lean theorems **with their proofs**, which Lean
re-checks in every file that cites them; the rest are Mathlib theorems under their textbook names. Run
the next cell to see any fact's exact statement.

In [2]:
for fact in [an.ivt, an.mvt, an.bolzano_weierstrass, an.triangle_inequality]:
    print(f"{fact.label}  [{fact.name}]")
    print(getattr(fact, "lean", "").strip() or "(a Mathlib theorem)", "\n")

the intermediate value theorem  [PyLeanX.Analysis.ivt]
theorem PyLeanX.Analysis.ivt {f : ℝ → ℝ} {a b y : ℝ} (hab : a ≤ b)
    (hf : ContinuousOn f (Set.Icc a b)) (ha : f a ≤ y) (hb : y ≤ f b) :
    ∃ c : ℝ, a ≤ c ∧ c ≤ b ∧ f c = y := by
  obtain ⟨c, ⟨h1, h2⟩, hc⟩ := intermediate_value_Icc hab hf ⟨ha, hb⟩
  exact ⟨c, h1, h2, hc⟩ 

the mean value theorem  [PyLeanX.Analysis.mvt]
theorem PyLeanX.Analysis.mvt {f : ℝ → ℝ} {a b : ℝ} (hab : a < b)
    (hf : ContinuousOn f (Set.Icc a b)) (hd : DifferentiableOn ℝ f (Set.Ioo a b)) :
    ∃ c : ℝ, a < c ∧ c < b ∧ deriv f c = (f b - f a) / (b - a) := by
  obtain ⟨c, ⟨h1, h2⟩, hc⟩ := exists_deriv_eq_slope f hab hf hd
  exact ⟨c, h1, h2, hc⟩ 

Bolzano–Weierstrass: a bounded sequence has a convergent subsequence  [PyLeanX.Analysis.bolzano_weierstrass]
theorem PyLeanX.Analysis.bolzano_weierstrass {a : ℕ → ℝ} {M : ℝ} (h : ∀ n, |a n| ≤ M) :
    ∃ φ : ℕ → ℕ, ∃ L : ℝ,
      StrictMono φ ∧ Filter.Tendsto (fun n => a (φ n)) Filter.atTop (nhds L) := by
  obtai

---
## 3. Worked examples

### Example 1. $\frac{1}{n+1} \to 0$

**Explore.** For each ε, the proof must find an N after which every term stays in the band $|y| < ε$.
Move the slider: the green terms are inside the band from N on.

In [3]:
show(figure_sequence()[0])

In [4]:
explore_sequence()

**The proof as in a book.** Let ε > 0. By the Archimedean property choose N ∈ ℕ with N > 1/ε. Let n ≥ N.
Then n + 1 > N > 1/ε, so 1/(n+1) < ε, and |1/(n+1) − 0| = 1/(n+1) < ε. ∎

**The same proof in PyLeanX** — one line per sentence:

In [5]:
example_1 = one_over_n(
    pl.suffices(eps_N(lambda n: 1 / (R(n) + 1), 0), because=an.tendsto_iff),
    pl.fix(eps, assume={"hε": eps > 0}),
    pl.choose(N, such_that=1 / eps < R(N), because=an.archimedean, names="hN"),
    pl.take(N),
    pl.fix(n, assume={"hn": n >= N}),
    pl.claim(R(N) <= R(n), because="hn", note="n ≥ N"),
    pl.claim(1 / eps < R(n) + 1, note="1/ε < N ≤ n < n + 1"),
    pl.claim(1 / (R(n) + 1) < eps, because=an.reciprocal_lt),
    pl.claim(1 / (R(n) + 1) > 0, name="hpos"),
    pl.chain(pl.abs(1 / (R(n) + 1) - 0),
             ("=", pl.abs(1 / (R(n) + 1))),
             ("=", 1 / (R(n) + 1), [an.abs_of_pos, "hpos"]),
             ("<", eps)),
)
check(example_1)

⏳ *Lean is checking this proof (the first check of a session takes a minute or more; a proof that was already checked comes back at once)…*

**Theorem** (`one_over_n_tends_to_zero`). $\frac{1}{n + 1} \to 0$

*Proof.*

1. It suffices to show $\forall \varepsilon \in \mathbb{R},\ \varepsilon > 0 \to \exists N \in \mathbb{N},\ \forall n \in \mathbb{N},\ n \geq N \to \left|\frac{1}{\left(n + 1} - 0\right| < \varepsilon\right)$. — *definition of a limit (ε–N)*
2. Let $ε \in \mathbb{R}$ with $\varepsilon > 0$.
3. Choose $N$ such that $\frac{1}{\varepsilon} < N$. — *the Archimedean property*
4. Take $N$.
5. Let $n \in \mathbb{N}$ with $n \geq N$.
6. $N \leq n$. — *n ≥ N*
7. $\frac{1}{\varepsilon} < n + 1$. — *1/ε < N ≤ n < n + 1*
8. $\frac{1}{n + 1} < \varepsilon$. — *taking reciprocals: 1/ε < x gives 1/x < ε*
9. $\frac{1}{n + 1} > 0$.
10. $\left|\frac{1}{n + 1} - 0\right| = \left|\frac{1}{n + 1}\right| = \frac{1}{n + 1} \quad \text{(|x| = x for x > 0, by step 9)} < \varepsilon$.

Hence $\left|\frac{1}{n + 1} - 0\right| < \varepsilon$. ∎

✔ Checked by Lean (axioms: propext, Classical.choice, Quot.sound).

✅ **Lean accepted the proof.** Axioms used: `propext, Classical.choice, Quot.sound` (the standard ones).

✅ **one_over_n_tends_to_zero**: `verified` (cached)

axioms: `propext, Classical.choice, Quot.sound`

### Example 2. Limits are unique (a proof by contradiction)

If $a_n \to L$ and $a_n \to L'$ with $L \ne L'$, take $ε = |L - L'|/2$: eventually $a_n$ is within ε of
both, which is impossible.

In [6]:
show(figure_two_limits())

**In a book.** Suppose L ≠ L′, so |L − L′| > 0. Choose N₁ with |aₙ − L| < |L − L′|/2 for n ≥ N₁ and N₂ with
|aₙ − L′| < |L − L′|/2 for n ≥ N₂. For n = max(N₁, N₂),
|L − L′| ≤ |L − aₙ| + |aₙ − L′| < |L − L′|, a contradiction. ∎

In [7]:
M_ = pl.max(N1, N2)
example_2 = limits_unique(
    pl.claim(eps_N(a, L), because=[an.tendsto_iff, "hL"], name="hL_eps"),
    pl.claim(eps_N(a, L2), because=[an.tendsto_iff, "hL'"], name="hL'_eps"),
    pl.suppose_not(L != L2, name="hne"),
    pl.claim(pl.abs(L - L2) > 0, because=an.abs_pos),
    pl.choose(N1, such_that=eventually_close(N1, a, L, pl.abs(L - L2) / 2),
              because="hL_eps", names="hN₁", note="the definition with ε = |L − L'|/2"),
    pl.choose(N2, such_that=eventually_close(N2, a, L2, pl.abs(L - L2) / 2),
              because="hL'_eps", names="hN₂", note="the definition with ε = |L − L'|/2"),
    pl.claim(pl.abs(a(M_) - L) < pl.abs(L - L2) / 2, because="hN₁", note="max(N₁, N₂) ≥ N₁"),
    pl.claim(pl.abs(a(M_) - L2) < pl.abs(L - L2) / 2, because="hN₂", note="max(N₁, N₂) ≥ N₂"),
    pl.chain(pl.abs(L - L2),
             ("=", pl.abs((L - a(M_)) + (a(M_) - L2)), None, "add and subtract"),
             ("<=", pl.abs(L - a(M_)) + pl.abs(a(M_) - L2), an.triangle_inequality),
             ("<", pl.abs(L - L2) / 2 + pl.abs(L - L2) / 2, an.abs_sub_comm),
             ("=", pl.abs(L - L2))),
    pl.conclude(note="|L − L'| < |L − L'| is impossible"),
)
check(example_2)

⏳ *Lean is checking this proof (the first check of a session takes a minute or more; a proof that was already checked comes back at once)…*

**Theorem** (`limits_are_unique`). $\forall a \in \mathbb{N} \to \mathbb{R},\ L, L' \in \mathbb{R},\quad a_n \to L,\ a_n \to L' \;\Longrightarrow\; L = L'$

*Proof.*

1. $\forall \varepsilon \in \mathbb{R},\ \varepsilon > 0 \to \left(\exists N \in \mathbb{N},\ \forall n \in \mathbb{N},\ n \geq N \to \left|a\left(n\right) - L\right| < \varepsilon\right)$. — *definition of a limit (ε–N), aₙ → L (hL)*
2. $\forall \varepsilon \in \mathbb{R},\ \varepsilon > 0 \to \left(\exists N \in \mathbb{N},\ \forall n \in \mathbb{N},\ n \geq N \to \left|a\left(n\right) - L'\right| < \varepsilon\right)$. — *definition of a limit (ε–N), aₙ → L′ (hL')*
3. Suppose, for contradiction, that $L \neq L'$.
4. $\left|L - L'\right| > 0$. — *|x| > 0 for x ≠ 0*
5. Choose $N₁$ such that $\forall n \in \mathbb{N},\ n \geq N_{1} \to \left|a\left(n\right) - L\right| < \frac{\left|L - L'\right|}{2}$. — *the definition with ε = |L − L'|/2*
6. Choose $N₂$ such that $\forall n \in \mathbb{N},\ n \geq N_{2} \to \left|a\left(n\right) - L'\right| < \frac{\left|L - L'\right|}{2}$. — *the definition with ε = |L − L'|/2*
7. $\left|a\left(\max\left(N_{1}, N_{2}\right)\right) - L\right| < \frac{\left|L - L'\right|}{2}$. — *max(N₁, N₂) ≥ N₁*
8. $\left|a\left(\max\left(N_{1}, N_{2}\right)\right) - L'\right| < \frac{\left|L - L'\right|}{2}$. — *max(N₁, N₂) ≥ N₂*
9. $\left|L - L'\right| = \left|L - a\left(\max\left(N_{1}, N_{2}\right)\right) + \left(a\left(\max\left(N_{1}, N_{2}\right)\right) - L'\right)\right| \quad \text{(add and subtract)} \le \left|L - a\left(\max\left(N_{1}, N_{2}\right)\right)\right| + \left|a\left(\max\left(N_{1}, N_{2}\right)\right) - L'\right| \quad \text{(triangle inequality)} < \frac{\left|L - L'\right|}{2} + \frac{\left|L - L'\right|}{2} \quad \text{(|x − y| = |y − x|)} = \left|L - L'\right|$.

This is a contradiction (|L − L'| < |L − L'| is impossible). ∎

✔ Checked by Lean (axioms: propext, Classical.choice, Quot.sound).

✅ **Lean accepted the proof.** Axioms used: `propext, Classical.choice, Quot.sound` (the standard ones).

✅ **limits_are_unique**: `verified` (cached)

axioms: `propext, Classical.choice, Quot.sound`

### Example 3. $f(x) = 3x + 1$ is continuous (ε–δ)

**Explore.** For each ε, δ = ε/3 works: inside the yellow δ-strip the graph stays in the blue ε-strip.

In [8]:
show(figure_eps_delta()[0])

In [9]:
explore_eps_delta()

**In a book.** Let ε > 0 and take δ = ε/3 > 0. If |x − x₀| < δ then
|f(x) − f(x₀)| = |3(x − x₀)| = 3|x − x₀| < 3·ε/3 = ε. ∎

In [10]:
example_3 = linear_continuous(
    pl.suffices(eps_delta(f, x0), because=an.continuous_at_iff),
    pl.fix(eps, assume={"hε": eps > 0}),
    pl.take(eps / 3, such_that=eps / 3 > 0),
    pl.fix(x, assume={"hx": pl.abs(x - x0) < eps / 3}),
    pl.chain(pl.abs(f(x) - f(x0)),
             ("=", pl.abs(3 * (x - x0)), "hf", "f(x) = 3x + 1"),
             ("=", 3 * pl.abs(x - x0), [an.abs_mul, an.abs_of_pos]),
             ("<", 3 * (eps / 3)),
             ("=", eps)),
)
check(example_3)

⏳ *Lean is checking this proof (the first check of a session takes a minute or more; a proof that was already checked comes back at once)…*

**Theorem** (`three_x_plus_one_continuous`). $\forall f \in \mathbb{R} \to \mathbb{R},\ x_{0} \in \mathbb{R},\quad \forall x \in \mathbb{R},\ f\left(x\right) = 3 x + 1 \;\Longrightarrow\; f \text{ is continuous at } x_{0}$

*Proof.*

1. It suffices to show $\forall \varepsilon \in \mathbb{R},\ \varepsilon > 0 \to \left(\exists \delta \in \mathbb{R},\ \delta > 0 \land \left(\forall x \in \mathbb{R},\ \left|x - x_{0}\right| < \delta \to \left|f\left(x\right) - f\left(x_{0}\right)\right| < \varepsilon\right)\right)$. — *definition of continuity (ε–δ)*
2. Let $ε \in \mathbb{R}$ with $\varepsilon > 0$.
3. Take $δ = \frac{\varepsilon}{3}$; then $\frac{\varepsilon}{3} > 0$.
4. Let $x \in \mathbb{R}$ with $\left|x - x_{0}\right| < \frac{\varepsilon}{3}$.
5. $\left|f\left(x\right) - f\left(x_{0}\right)\right| = \left|3 \left(x - x_{0}\right)\right| \quad \text{(f(x) = 3x + 1)} = 3 \left|x - x_{0}\right| \quad \text{(|xy| = |x||y|, |x| = x for x > 0)} < 3 \cdot \frac{\varepsilon}{3} = \varepsilon$.

Hence $\left|f\left(x\right) - f\left(x_{0}\right)\right| < \varepsilon$. ∎

✔ Checked by Lean (axioms: propext, Classical.choice, Quot.sound).

✅ **Lean accepted the proof.** Axioms used: `propext, Classical.choice, Quot.sound` (the standard ones).

✅ **three_x_plus_one_continuous**: `verified` (cached)

axioms: `propext, Classical.choice, Quot.sound`

### Example 4. Citing the big theorems: a bounded sequence has a Cauchy subsequence

A research-style proof is mostly *citations*. Bolzano–Weierstrass gives a convergent subsequence, and a
convergent sequence is Cauchy:

In [11]:
example_4 = bounded_cauchy_subsequence(
    pl.choose([phi, L], such_that=pl.And(an.strictly_increasing(phi),
                                         an.tendsto(pl.fun(lambda n: a(phi(n)), pl.Nat), L)),
              because=[an.bolzano_weierstrass, "hM"], names="hφ hlim"),
    pl.take(phi, such_that=an.strictly_increasing(phi), because="hφ"),
    pl.conclude(because=[an.convergent_cauchy, "hlim"]),
)
check(example_4)

⏳ *Lean is checking this proof (the first check of a session takes a minute or more; a proof that was already checked comes back at once)…*

**Theorem** (`bounded_sequence_has_cauchy_subsequence`). $\forall a \in \mathbb{N} \to \mathbb{R},\ M \in \mathbb{R},\quad \forall n \in \mathbb{N},\ \left|a\left(n\right)\right| \leq M \;\Longrightarrow\; \exists \varphi \in \mathbb{N} \to \mathbb{N},\ \varphi \text{ is strictly increasing} \land \left(a\left(\varphi\left(n\right)\right)\right)_{n} \text{ is Cauchy}$

*Proof.*

1. Choose $φ, L$ such that $\varphi \text{ is strictly increasing} \land a\left(\varphi\left(n\right)\right) \to L$. — *Bolzano–Weierstrass: a bounded sequence has a convergent subsequence, ∀ n ∈ ℕ, |a(n)| ≤ M (hM)*
2. Take $\varphi$; then $\varphi \text{ is strictly increasing}$. — *φ is strictly increasing (hφ)*

Hence $\left(a\left(\varphi\left(n\right)\right)\right)_{n} \text{ is Cauchy}$ (a convergent sequence is Cauchy, a(φ(n)) → L (hlim)). ∎

✔ Checked by Lean (axioms: propext, Classical.choice, Quot.sound).

✅ **Lean accepted the proof.** Axioms used: `propext, Classical.choice, Quot.sound` (the standard ones).

✅ **bounded_sequence_has_cauchy_subsequence**: `verified` (cached)

axioms: `propext, Classical.choice, Quot.sound`

### Example 5. Building on your own results

A theorem you proved can be cited like any library fact: `because=example_1`. Lean then checks the cited
proof together with the new one (a theorem whose proof was never given cannot be cited).

In [12]:
two_over_n = pl.Theorem(
    "two_over_n_tends_to_zero", [],
    conclusion=an.tendsto(pl.fun(lambda n: 2 * (1 / (R(n) + 1)), pl.Nat), 0),
    proof=pl.proof(
        pl.claim(an.tendsto(pl.fun(lambda n: 2 * (1 / (R(n) + 1)), pl.Nat), pl.real(2) * 0),
                 because=[an.limit_const_mul, example_1], note="by Example 1"),
        pl.conclude(note="2 · 0 = 0"),
    ),
)
check(two_over_n)

⏳ *Lean is checking this proof (the first check of a session takes a minute or more; a proof that was already checked comes back at once)…*

**Theorem** (`two_over_n_tends_to_zero`). $2 \cdot \frac{1}{n + 1} \to 0$

*Proof.*

1. $2 \cdot \frac{1}{n + 1} \to 2 \cdot 0$. — *by Example 1*

Hence $2 \cdot \frac{1}{n + 1} \to 0$ (2 · 0 = 0). ∎

✔ Checked by Lean (axioms: propext, Classical.choice, Quot.sound).

✅ **Lean accepted the proof.** Axioms used: `propext, Classical.choice, Quot.sound` (the standard ones).

✅ **two_over_n_tends_to_zero**: `verified` (cached)

axioms: `propext, Classical.choice, Quot.sound`

---
## 4. When a step is wrong

A classic slip in Example 3: taking δ = ε. Everything else is unchanged.

In [13]:
wrong = linear_continuous(
    pl.suffices(eps_delta(f, x0), because=an.continuous_at_iff),
    pl.fix(eps, assume={"hε": eps > 0}),
    pl.take(eps, such_that=eps > 0),                       # ✗ δ = ε is too large
    pl.fix(x, assume={"hx": pl.abs(x - x0) < eps}),
    pl.chain(pl.abs(f(x) - f(x0)),
             ("=", pl.abs(3 * (x - x0)), "hf", "f(x) = 3x + 1"),
             ("=", 3 * pl.abs(x - x0), [an.abs_mul, an.abs_of_pos]),
             ("<", eps)),
)
check(wrong)

⏳ *Lean is checking this proof (the first check of a session takes a minute or more; a proof that was already checked comes back at once)…*

**Theorem** (`three_x_plus_one_continuous`). $\forall f \in \mathbb{R} \to \mathbb{R},\ x_{0} \in \mathbb{R},\quad \forall x \in \mathbb{R},\ f\left(x\right) = 3 x + 1 \;\Longrightarrow\; f \text{ is continuous at } x_{0}$

*Proof.*

1. It suffices to show $\forall \varepsilon \in \mathbb{R},\ \varepsilon > 0 \to \left(\exists \delta \in \mathbb{R},\ \delta > 0 \land \left(\forall x \in \mathbb{R},\ \left|x - x_{0}\right| < \delta \to \left|f\left(x\right) - f\left(x_{0}\right)\right| < \varepsilon\right)\right)$. — *definition of continuity (ε–δ)*
2. Let $ε \in \mathbb{R}$ with $\varepsilon > 0$.
3. Take $δ = \varepsilon$; then $\varepsilon > 0$.
4. Let $x \in \mathbb{R}$ with $\left|x - x_{0}\right| < \varepsilon$.
5. $\left|f\left(x\right) - f\left(x_{0}\right)\right| = \left|3 \left(x - x_{0}\right)\right| \quad \text{(f(x) = 3x + 1)} = 3 \left|x - x_{0}\right| \quad \text{(|xy| = |x||y|, |x| = x for x > 0)} < \varepsilon$.

Hence $\left|f\left(x\right) - f\left(x_{0}\right)\right| < \varepsilon$. ∎

✘ Lean did not accept this proof (rejected).

❌ **Lean did not accept this proof.**

Lean could not justify **step 5** from the reasons given.

Lean's own words: `unsolved goals`

Check that the step is really true at that point of the proof and that the cited facts (with the earlier steps) give it.

❌ **three_x_plus_one_continuous**: `rejected`

- warning: `aesop: failed to prove the goal after exhaustive search.` (step 5: have step5 : abs (f x - f x₀) < ε := by)

- error: `unsolved goals` (step 5: have step5 : abs (f x - f x₀) < ε := by)

Lean names the step: from |x − x₀| < ε it does not follow that 3|x − x₀| < ε. Nothing is accepted
because it "looks right".

---
## 5. Exercises

Each exercise fixes the statement; write the proof inside the call and run the cell. The hints are
collapsed; the solutions are at the end.

### Exercise 1. If $a_n \to L$ then $a_{n+1} \to L$.

*Given:* `hL`: $a_n \to L$.

<details><summary><b>Hint</b></summary>

Unfold the hypothesis with `an.tendsto_iff` (a `pl.claim` of `eps_N(a, L)`), then it suffices to show
`eps_N(lambda n: a(n + 1), L)`. Given ε, the *same* N works: if n ≥ N then n + 1 ≥ N.
</details>

In [14]:
my_proof_1 = exercise_1(
    # pl.claim(eps_N(a, L), because=[an.tendsto_iff, "hL"], name="hL_eps"),
    # ...
)
check(my_proof_1)

✏️ *Write your steps first:* put the steps of your proof inside the exercise call, then run the cell again.

### Exercise 2. If $a_n \to L$ and $L > 0$, then $a_n > 0$ for all large $n$.

*Given:* `hL`: $a_n \to L$; `hpos`: $L > 0$. *Show:* ∃N ∀n ≥ N, aₙ > 0.

<details><summary><b>Hint 1</b></summary>

Use the definition with ε = L/2: `pl.choose(N, such_that=eventually_close(N, a, L, L / 2), because="hL_eps")`.
</details>
<details><summary><b>Hint 2</b></summary>

From |aₙ − L| < L/2, the fact `an.abs_lt` gives −L/2 < aₙ − L < L/2; then aₙ > L/2 > 0 (Lean does this last
step itself).
</details>

In [15]:
my_proof_2 = exercise_2(
    # ...
)
check(my_proof_2)

✏️ *Write your steps first:* put the steps of your proof inside the exercise call, then run the cell again.

### Exercise 3. $x^3 + x - 1$ has a root in $[0, 1]$.

*Show:* ∃ c, 0 ≤ c ≤ 1 and c³ + c − 1 = 0.

In [16]:
show(figure_ivt())

<details><summary><b>Hint</b></summary>

`pl.let(p, lambda t: t**3 + t - 1)`, then claim that p is continuous on [0, 1], that p(0) ≤ 0 and
0 ≤ p(1); choose c by `an.ivt` and take it.
</details>

In [17]:
my_proof_3 = exercise_3(
    # pl.let(p, lambda t: t**3 + t - 1),
    # ...
)
check(my_proof_3)

✏️ *Write your steps first:* put the steps of your proof inside the exercise call, then run the cell again.

### Exercise 4. A convergent sequence is Cauchy.

*Given:* `hL`: $a_n \to L$. *Show:* $(a_n)$ is Cauchy — from the definitions, without citing
`an.convergent_cauchy`.

<details><summary><b>Hint</b></summary>

Unfold with `an.cauchy_iff` (it suffices to show `cauchy_eps(a)`). Use ε/2 in the definition of the limit,
fix m ≥ N and n ≥ N, and chain |aₘ − aₙ| = |(aₘ − L) + (L − aₙ)| ≤ |aₘ − L| + |L − aₙ| < ε/2 + ε/2 = ε.
</details>

In [18]:
my_proof_4 = exercise_4(
    # ...
)
check(my_proof_4)

✏️ *Write your steps first:* put the steps of your proof inside the exercise call, then run the cell again.

---
## 6. Writing your own theorem

```python
my_theorem = pl.Theorem(
    "name_of_the_result",
    [a, L],                                   # the variables
    assumptions={"hL": an.tendsto(a, L)},     # named hypotheses
    conclusion=an.tendsto(pl.fun(lambda n: 2 * a(n), pl.Nat), 2 * L),
    proof=pl.proof(
        pl.conclude(because=[an.limit_const_mul, "hL"]),
    ),
)
check(my_theorem)
```

New symbols: `pl.symbol("name", pl.Real)`, sequences `pl.Fn(pl.Nat, pl.Real)`, functions
`pl.Fn(pl.Real, pl.Real)`. When Lean cannot fill a gap, split the step into smaller ones or cite the fact
it needs. **Not in the catalogue yet:** series, Riemann integrals, sequences of functions and uniform
convergence, lim sup / lim inf, general metric spaces — Mathlib has them; they still need textbook-style
vocabulary here.

---
## 7. Solutions (each checked by Lean)

Try first! Each solution was verified by Lean with the standard axioms only. Run `show_solution(1)` …
`show_solution(4)` to check one again.

<details><summary><b>Solution 1</b></summary>

```python
exercise_1(
    pl.claim(eps_N(a, L), because=[an.tendsto_iff, "hL"], name="hL_eps"),
    pl.suffices(eps_N(lambda n: a(n + 1), L), because=an.tendsto_iff),
    pl.fix(eps, assume={"hε": eps > 0}),
    pl.choose(N, such_that=eventually_close(N, a, L, eps), because="hL_eps", names="hN"),
    pl.take(N),
    pl.fix(n, assume={"hn": n >= N}),
    pl.conclude(because="hN", note="n + 1 ≥ N"),
)
```
</details>

<details><summary><b>Solution 2</b></summary>

```python
exercise_2(
    pl.claim(eps_N(a, L), because=[an.tendsto_iff, "hL"], name="hL_eps"),
    pl.choose(N, such_that=eventually_close(N, a, L, L / 2), because="hL_eps", names="hN",
              note="the definition with ε = L/2"),
    pl.take(N),
    pl.fix(n, assume={"hn": n >= N}),
    pl.claim(pl.abs(a(n) - L) < L / 2, because="hN", note="by the choice of N"),
    pl.claim(pl.And(-(L / 2) < a(n) - L, a(n) - L < L / 2), because=an.abs_lt),
    pl.conclude(note="aₙ > L − L/2 = L/2 > 0"),
)
```
</details>

<details><summary><b>Solution 3</b></summary>

```python
exercise_3(
    pl.let(p, lambda t: t**3 + t - 1),
    pl.claim(an.continuous_on(p, 0, 1), note="polynomials are continuous"),
    pl.claim(p(0) <= 0, note="p(0) = −1"),
    pl.claim(0 <= p(1), note="p(1) = 1"),
    pl.choose(c, such_that=pl.And(0 <= c, c <= 1, p(c) == 0), because=an.ivt,
              names="h0c hc1 hpc"),
    pl.take(c),
    pl.conclude(note="p(c) = c³ + c − 1"),
)
```
</details>

<details><summary><b>Solution 4</b></summary>

```python
exercise_4(
    pl.claim(eps_N(a, L), because=[an.tendsto_iff, "hL"], name="hL_eps"),
    pl.suffices(cauchy_eps(a), because=an.cauchy_iff),
    pl.fix(eps, assume={"hε": eps > 0}),
    pl.choose(N, such_that=eventually_close(N, a, L, eps / 2), because="hL_eps", names="hN"),
    pl.take(N),
    pl.fix(m, assume={"hm": m >= N}),
    pl.fix(n, assume={"hn": n >= N}),
    pl.claim(pl.abs(a(m) - L) < eps / 2, because="hN", note="m ≥ N"),
    pl.claim(pl.abs(a(n) - L) < eps / 2, because="hN", note="n ≥ N"),
    pl.chain(pl.abs(a(m) - a(n)),
             ("=", pl.abs((a(m) - L) + (L - a(n))), None, "add and subtract"),
             ("<=", pl.abs(a(m) - L) + pl.abs(L - a(n)), an.triangle_inequality),
             ("<", eps / 2 + eps / 2, an.abs_sub_comm),
             ("=", eps)),
)
```
</details>